# US County Business Patterns: Economic Analysis (2017–2023)

## Dataset
The **Census Bureau County Business Patterns (CBP)** API provides annual employment, payroll and establishment counts broken down by:

- **Geography**: all US counties and states

- **Sector**: 19 NAICS industry sectors.

- **Business size**: 9 employee-count sizes.

- **Legal form** (state level only): corporate, sole proprietorship, nonprofit, government.

They cover the 2017–2023 7 years period, giving us a window that spans from pre-COVID to COVID shock and recovery.

## Table of Contents
- [Setting up the enviroment](#set-env)
    - [1. Virtual Enviroment](#venv)
    - [2. Other Useful Packages](#add-pack)
    - [3. Set Up Google Cloud CLI](#gcloud)

- [Data Retreival](#data)
    - [1. Collection Script](#coll)
    - [2. Merge Script](#merge)
    - [3. Resulting output](#res-out)

- [Setting up the Cloud](#cloud-env)
    - [1. Create a Bucket](#buck)
    - [2. Upload Data to Cloud](#gcloud-up)
    - [3. Create Dataset and Tables](#gcloud-data-tab)

- [Queries](#queries)
    - [1. Preparations](#prep)
    - [2. Query 1](#q1)
    - [3. Query 2](#q2)
    - [4. Query 3](#q3)
    - [5. Query 4](#q4)
    - [6. Query 5](#q5)

- [Machine Learning Pipeline](#MLintro)

- [Conclusion](#conc)

## <a id="set-env">Setting up the environment</a>

### <a id="venv">Virtual Enviroment</a>

The repository already provides what's necessary to set up the virtual environment.

To use the .toml file and uv.lock, the uv package manager needs to be installed.

For Ubuntu-based machines (or any Linux machines that have access to snap) run the following:

```bash
 snap install astral-uv --classic
```
After that, run:

```bash
uv sync
```

For standard pip users, a `requirements.txt` is also provided.

```bash
python venv -m .venv
source .venv/bin/activate # For Windows .venv/Scripts/activate.bat
pip install -r requirements.txt
```

### <a id="other-pack">Other Useful Packages</a>

Other non-python-related packages are required to run this notebook and the tools used in it.

The commands to use, in a debian-based machine, are:

```bash
PACKAGES=(
  "openjdk-17-jre" # java version 17. For spark, if runned locally
  "bat"            # better cat
)

apt-get install "${PACKAGES[@]}" -y
```

### <a id="gcloud">Set up Google Cloud CLI</a>

In order to use the following notebook to perform queries and analysis on a project present in GCP, we need to to set up the google cloud connection first.

For doing that, we need need the gcloud-cli package present in snap, with the following command:
```bash
snap install google-cloud-cli --classic
```

To log Current Machine:

```bash
!gcloud auth login --no-launch-browser
```

In [1]:
# Auth the Notebook
from google.cloud import bigquery
from google.oauth2 import service_account
import os

PROJECT_ID = !gcloud projects list | grep ccbd-exam | tr ' ' '\n' | head -n 1
PROJECT_ID = PROJECT_ID[0]

key_path = os.environ['GOOGLE_APPLICATION_CREDENTIALS'] 

credentials = service_account.Credentials.from_service_account_file(key_path, scopes=['https://www.googleapis.com/auth/cloud-platform'])
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

print('Notebook ready for project:', client.project)

Notebook ready for project: ccbd-exam-2026-maurov


## <a id="data">Data Retreival</a>

Our data come from the American **Census Bureau County Business Patterns (CBP)**, a colleciton of several panel data concerning different economic metrices.

In order to collect them and merge them in csv that we can put in a bucket, we need to use the following scripts.

In [ ]:
!ls -l . --color | grep .sh

# They already are executable, no need to chmod them. If that was not the case, we can run
# !chmod +x *.sh
# to change the executability permission for all '.sh' files in our repository

-rwxrwxr-x 1 mauro mauro      960 Jun 13 16:07 collect.sh
-rwxrwxr-x 1 mauro mauro      614 Jun 11 17:15 merger.sh


### <a id="coll"> Collection Script</a>

The collection script:
1. Loads the API key needed to gather data from the census database using an `.env` file.
2. Creates the raw data output directory.
3. Runs a curl GET request for both county and state level data, saving the outputs in `OUTPUT_DIR`.
4. Repeats step 3 for all specified NAICS sectors across all years between 2017-2023 (fetching all US counties and states via wildcards).

The data we collect are the following:

#### For County Level
| Get-Column | Meaning |
|---| --- |
|**NAME**| Name of the County|
|**NAICS2017_LABEL**| Code Label of the Sector referred to the NAICS code|
|**EMP**| Total number of employees |
|**PAYANN**| Sum of Annual Payroll |
|**ESTAB**| Total number of Businesses |

#### For State Level
| Get-Column | Meaning |
|---| --- |
|**NAME**| Name of the State|
|**NAICS2017_LABEL**| Code Label of the Sector referred to the NAICS code|
|**EMP**| Total number of employees |
|**PAYANN**| Sum of Annual Payroll |
|**ESTAB**| Total number of Businesses |
|**EMPSZES**| Code of size of the Businesses |
|**EMPSZES_LABEL**| Description of EMPSZES |
|**LFO**| Code of Legal Form of Organization |
|**LFO_LABEL**| Description of LFO |

**The sectors we are pulling are:**

| Sector Code | Official 2017 NAICS Description |
| --- | --- |
| **00** | Total for State (All Sectors) |
| **11** | Agriculture, Forestry, Fishing and Hunting |
| **21** | Mining, Quarrying, and Oil and Gas Extraction |
| **22** | Utilities |
| **23** | Construction |
| **31-33** | Manufacturing |
| **42** | Wholesale Trade |
| **44-45** | Retail Trade |
| **48-49** | Transportation and Warehousing |
| **51** | Information |
| **52** | Finance and Insurance |
| **53** | Real Estate and Rental and Leasing |
| **54** | Professional, Scientific, and Technical Services |
| **56** | Administrative and Support and Waste Management and Remediation Services |
| **61** | Educational Services |
| **62** | Health Care and Social Assistance |
| **71** | Arts, Entertainment, and Recreation |
| **72** | Accommodation and Food Services |
| **81** | Other Services (except Public Administration) |

In [6]:
!batcat ./collect.sh

───────┬────────────────────────────────────────────────────────────────────────
       │ File: ./collect.sh
───────┼────────────────────────────────────────────────────────────────────────
   1   │ #!/bin/bash
   2   │ 
   3   │ set -a
   4   │ source .env
   5   │ set +a
   6   │ 
   7   │ if [ -z "$API_KEY" ]; then
   8   │   echo "Error: API_KEY not set"
   9   │   exit 1
  10   │ fi
  11   │ 
  12   │ BASE_URL="https://api.census.gov/data"
  13   │ OUTPUT_DIR="./data_raw"
  14   │ 
  15   │ mkdir -p "$OUTPUT_DIR"
  16   │ 
  17   │ SECTORS=("00" "11" "21" "22" "23" "31-33" "42" "44-45" "48-49" "51" "52
       │ " "53" "54" "56" "61" "62" "71" "72" "81")
  18   │ 
  19   │ for YEAR in $(seq 2017 2023); do
  20   │   for SECTOR in "${SECTORS[@]}"; do
  21   │     curl -s "${BASE_URL}/${YEAR}/cbp?get=NAME,NAICS2017_LABEL,EMP,PAYAN
       │ N,ESTAB&for=county:*&NAICS2017=${SECTOR}&key=${API_KEY}" \
  22   │       -o "${OUTPUT_DIR}/cbp_county_${SECTOR}_${YEAR}.json"
  23   │     sleep 

The file we retreive are several json files

In [ ]:
# Number of files we retreived
!ls ./data_raw/ | wc -l

266


In [10]:
!ls ./data_raw/ | head -n 5

cbp_county_00_2017.json
cbp_county_00_2018.json
cbp_county_00_2019.json
cbp_county_00_2020.json
cbp_county_00_2021.json


In [11]:
!du -hs ./data_raw/

82M	./data_raw/


### <a id="merge"> Merge Script</a>

The merging script processes the raw JSON files collected using `jq` cli utility, by extracting the header, adding a  temporal marker (`year`) to and, append data iteratively, creating two structured CSV files (one for county level and one for state level data).

#### Steps

1. Extracts the first element (header) from the first county JSON file, appends a new `"year"` column header, and create with it the initial line of `cbp_county.csv`.
   - We repeats this process for the state-level data to create `cbp_state.csv`.
2. Iterate through every individual JSON file found in the `data_raw/` directory.
   - For each file, we use a regex to extract the year directly from the name of the json file
3. With `jq` we skip the original header (`.[1:]`), append the extracted year string (`$y`) to the end of every row, convert the records to standard CSV format (`@csv`), and append the result to the respective CSV file.

In [13]:
!jq

jq - commandline JSON processor [version 1.8.1]

Usage:	jq [options] <jq filter> [file...]
	jq [options] --args <jq filter> [strings...]
	jq [options] --jsonargs <jq filter> [JSON_TEXTS...]

jq is a tool for processing JSON inputs, applying the given filter to
its JSON text inputs and producing the filter's results as JSON on
standard output.

The simplest filter is ., which copies jq's input to its output
unmodified except for formatting. For more advanced filters see
the jq(1) manpage ("man jq") and/or https://jqlang.org/.

Example:

	$ echo '{"foo": 0}' | jq .
	{
	  "foo": 0
	}

For listing the command options, use jq --help.


In [34]:
!batcat ./merger.sh

───────┬────────────────────────────────────────────────────────────────────────
       │ File: ./merger.sh
───────┼────────────────────────────────────────────────────────────────────────
   1   │ #!/bin/bash
   2   │ 
   3   │ jq -r '.[0] + ["year"] | @csv' $(ls data_raw/cbp_county_*.json | head -
       │ 1) >cbp_county.csv
   4   │ jq -r '.[0] + ["year"] | @csv' $(ls data_raw/cbp_state_*.json | head -1
       │ ) >cbp_state.csv
   5   │ 
   6   │ for f in data_raw/cbp_county_*.json; do
   7   │ 
   8   │   year=$(echo $f | grep -o '[0-9]\{4\}')
   9   │   jq -r --arg y "$year" '.[1:][] | . + [$y] | @csv' "$f"
  10   │ 
  11   │ done >>cbp_county.csv
  12   │ 
  13   │ for f in data_raw/cbp_state_*.json; do
  14   │ 
  15   │   year=$(echo $f | grep -o '[0-9]\{4\}')
  16   │   jq -r --arg y "$year" '.[1:][] | . + [$y] | @csv' "$f"
  17   │ 
  18   │ done >>cbp_state.csv
  19   │ 
  20   │ echo -e "Done:\n$(wc -l <cbp_county.csv) rows for county\n$(wc -l <cbp_
       │ state.csv) row

### <a id="res-out">Resulting Output</a>

To get, collect and merge the data we can do

```bash
./collect.sh && ./merger.sh
```

In [2]:
!batcat cbp_county.csv -l csv | head -n 10

"NAME","NAICS2017_LABEL","EMP","PAYANN","ESTAB","NAICS2017","state","county","year"
"Covington County, Alabama","Total for all sectors","10433","357563","816","00","01","039","2017"
"Autauga County, Alabama","Total for all sectors","11036","360936","869","00","01","001","2017"
"Baldwin County, Alabama","Total for all sectors","61792","2087497","5384","00","01","003","2017"
"Barbour County, Alabama","Total for all sectors","6834","237076","455","00","01","005","2017"
"Bibb County, Alabama","Total for all sectors","3484","136216","284","00","01","007","2017"
"Blount County, Alabama","Total for all sectors","6645","215192","698","00","01","009","2017"
"Bullock County, Alabama","Total for all sectors","1978","63818","107","00","01","011","2017"
"Butler County, Alabama","Total for all sectors","5983","184588","422","00","01","013","2017"
"Calhoun County, Alabama","Total for all sectors","36301","1247795","2319","00","01","015","2017"


In [1]:
!batcat cbp_state.csv -l csv | head -n 10

"NAME","NAICS2017_LABEL","EMP","PAYANN","ESTAB","EMPSZES","EMPSZES_LABEL","LFO","LFO_LABEL","NAICS2017","state","year"
"Mississippi","Total for all sectors","939485","35366718","59294","001","All establishments","001","All establishments","00","28","2017"
"Mississippi","Total for all sectors","54686","2144622","29629","210","Establishments with less than 5 employees","001","All establishments","00","28","2017"
"Mississippi","Total for all sectors","83999","2722937","12705","220","Establishments with 5 to 9 employees","001","All establishments","00","28","2017"
"Mississippi","Total for all sectors","111465","3702121","8333","230","Establishments with 10 to 19 employees","001","All establishments","00","28","2017"
"Mississippi","Total for all sectors","168553","5536516","5569","241","Establishments with 20 to 49 employees","001","All establishments","00","28","2017"
"Mississippi","Total for all sectors","121343","4134644","1783","242","Establishments with 50 to 99 employees","001","All e

In [31]:
!wc -l *.csv

  365616 cbp_county.csv
  314033 cbp_state.csv
  679649 total


## <a id="cloud-env">Setting Up The Cloud</a>

### <a id="buck">Create a Bucket</a>

First we need to set the project as primary project with 
```bash
gcloud config set project project-id
```
To check which project is currently the primary one we can run

```bash
gcloud config get project
```
To create a bucket for our main project we can use

```bash
gs://bucket-name --location=location-wanted --uniform-bucket-level-access
```


In [2]:
# We check our buckets and save the one we need

BUCKET_PATH = !gcloud storage ls
print(BUCKET_PATH)
BUCKET_PATH = BUCKET_PATH[0]

['gs://ccbd-exam-2026-maurov-bucket/']


### <a id="gcloud-up">Upload Data on GCP</a>

```bash
gcloud config set storage/parallel_composite_upload_enabled False
gcloud storage cp cbp_county.csv cbp_state.csv gs://ccbd-exam-2026-maurov-bucket/
```

In [ ]:
!gcloud storage ls gs://ccbd-exam-2026-maurov-bucket/

gs://ccbd-exam-2026-maurov-bucket/cbp_county.csv
gs://ccbd-exam-2026-maurov-bucket/cbp_state.csv


### <a id="gcloud-data-tab">Create Dataset and Tables</a>

To create the dataset we can use 
```bash
bq mk --location=europe-west8 cbp_dataset
```
Location of dataset and bucket where we put our data should match. We can check the latter with

```bash
gcloud storage buckets describe gs://ccbd-exam-2026-maurov-bucket --format="value(location)"  
```
Output: `EUROPE-WEST8`

#### Load county data
```bash
bq load \
  --location=europe-west8 \
  --source_format=CSV \
  --skip_leading_rows=1 \
  --schema="county_str:STRING,sector_label:STRING,num_employees:INT64,annual_payroll:FLOAT64,num_businesses:INT64,sector_code:STRING,state:STRING,county_code:STRING,year:INT64" \
  cbp_dataset.cbp_county \
  gs://ccbd-exam-2026-maurov-bucket/cbp_county.csv
```

#### Load state data

```bash
%%bash
bq load \
  --location=europe-west8 \
  --source_format=CSV \
  --skip_leading_rows=1 \
  --schema="state_str:STRING,sector_label:STRING,num_employees:INT64,annual_payroll:FLOAT64,num_businesses:INT64,bsize_code:STRING,bsize:STRING,lfo:STRING,lfo_label:STRING,sector_code:STRING,state:STRING,year:INT64" \
  cbp_dataset.cbp_state \
  gs://ccbd-exam-2026-maurov-bucket/cbp_state.csv
```

We are manually inferring the schema instead of doing it automatically, in this way we can rename the variables to be easier to work with, and cast the data type in a more consistent way.

This allow us to avoid using `create or replace table` later on, wasting time/computation.

In [ ]:
# We can check if tables were created by using
!bq ls cbp_dataset

   tableId     Type    Labels   Time Partitioning   Clustered Fields  
 ------------ ------- -------- ------------------- ------------------ 
  cbp_county   TABLE                                                  
  cbp_state    TABLE                                                  


In [2]:
# Set up SQL functions

from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

SQL_DIR = Path('sql')
DATASET='cbp_dataset'

def load_sql(name):
    return (SQL_DIR / f'{name}.sql').read_text()

def run_query(sql, max_gb=2.0, preview=True):
    if preview:
        dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        print(f'dry-run: will scan {dry.total_bytes_processed / 1e9:.3f} GB')
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(max_gb * 1e9))
    return client.query(sql, job_config=cfg).to_dataframe()

def show_and_run(name, max_gb=2.0):
    """Display the SQL from sql/<name> (the single source of truth), then run it."""
    sql = load_sql(name)
    display(Markdown(f"**`sql/{name}`**\n\n```sql\n{sql}\n```"))
    return run_query(sql, max_gb=max_gb)

In [17]:
run_query(f'''
select * 
from {DATASET}.cbp_state
limit 5
''')

dry-run: will scan 0.050 GB


,state_str,sector_label,num_employees,annual_payroll,num_businesses,bsize_code,bsize,lfo,lfo_label,sector_code,state,year
0,Mississippi,Total for all sectors,939485,35366718.0,59294,001,All establishments,001,All establishments,00,28,2017
1,Mississippi,Total for all sectors,54686,2144622.0,29629,210,Establishments with less than 5 employees,001,All establishments,00,28,2017
2,Mississippi,Total for all sectors,83999,2722937.0,12705,220,Establishments with 5 to 9 employees,001,All establishments,00,28,2017
3,Mississippi,Total for all sectors,111465,3702121.0,8333,230,Establishments with 10 to 19 employees,001,All establishments,00,28,2017
4,Mississippi,Total for all sectors,168553,5536516.0,5569,241,Establishments with 20 to 49 employees,001,All establishments,00,28,2017


In [6]:
run_query(f'''
select * 
from {DATASET}.cbp_county
limit 5
''')

dry-run: will scan 0.037 GB


,county_str,sector_label,num_employees,annual_payroll,num_businesses,sector_code,state,county_code,year
0,"Covington County, Alabama",Total for all sectors,10433,357563.0,816,00,01,039,2017
1,"Autauga County, Alabama",Total for all sectors,11036,360936.0,869,00,01,001,2017
2,"Baldwin County, Alabama",Total for all sectors,61792,2087497.0,5384,00,01,003,2017
3,"Barbour County, Alabama",Total for all sectors,6834,237076.0,455,00,01,005,2017
4,"Bibb County, Alabama",Total for all sectors,3484,136216.0,284,00,01,007,2017


## <a id="queries">Queries</a>

### <a id="prep">Preparations</a>

**Core Derived Metric: Pay Ratio**

To compare counties fairly across states of different sizes and wealth levels, we normalize county-level average pay against the state average for the same sector and year:

$$\text{pay\_ratio} = \frac{\text{avg\_pay\_county}} {\text{avg\_pay\_state}}$$

- if `pay_ratio > 1`: county pays **above** the state average for that sector

- if `pay_ratio < 1`: county pays **below**

- if `pay_ratio = 2.0`: county pays twice the state average

This normalization removes the effect of state-level wealth differences (comparing a Californian county to a Mississippi one directly would be misleading).

To get this result we make a new table `ratio_df`, joining `county_agg` (county-level averages) with `state_agg` (state benchmarks) on `(state, sector_code, year)`.

In [ ]:
agg_view = f'''
create or replace view {DATASET}.county_agg as (
  select
    county_str,
    county_code,
    state,
    sector_code,
    sector_label,
    year,
    num_employees,
    annual_payroll,
    num_businesses,
    annual_payroll / nullif(num_employees, 0) as avg_pay_ct
  from {DATASET}.cbp_county
);
'''
client.query(agg_view).result() 
print("View created successfully!")

View created successfully!


In [19]:
run_query(f'''
select * from {DATASET}.county_agg
limit 10;
'''
)

dry-run: will scan 0.037 GB


,county_str,county_code,state,sector_code,sector_label,year,num_employees,annual_payroll,num_businesses,avg_pay_ct
0,"Covington County, Alabama",039,01,00,Total for all sectors,2017,10433,357563.0,816,34.272309
1,"Autauga County, Alabama",001,01,00,Total for all sectors,2017,11036,360936.0,869,32.705328
2,"Baldwin County, Alabama",003,01,00,Total for all sectors,2017,61792,2087497.0,5384,33.782642
3,"Barbour County, Alabama",005,01,00,Total for all sectors,2017,6834,237076.0,455,34.690664
4,"Bibb County, Alabama",007,01,00,Total for all sectors,2017,3484,136216.0,284,39.097589
5,"Blount County, Alabama",009,01,00,Total for all sectors,2017,6645,215192.0,698,32.384048
6,"Bullock County, Alabama",011,01,00,Total for all sectors,2017,1978,63818.0,107,32.263903
7,"Butler County, Alabama",013,01,00,Total for all sectors,2017,5983,184588.0,422,30.852081
8,"Calhoun County, Alabama",015,01,00,Total for all sectors,2017,36301,1247795.0,2319,34.373571
9,"Chambers County, Alabama",017,01,00,Total for all sectors,2017,6899,226403.0,554,32.816785


In [ ]:
agg_view = f'''
create or replace view {DATASET}.state_agg as (
  select 
    state_str,
    state,
    year,
    sector_code,
    sector_label,
    sum(annual_payroll)  as total_pay_st,
    sum(num_employees)   as total_emp_st,
    sum(num_businesses)  as total_bs_st,
    sum(annual_payroll) / nullif(sum(num_employees), 0) as avg_pay_st
  from {DATASET}.cbp_state
  where lfo = '001'
    and bsize_code = '001'
  group by state_str, state, year, sector_code, sector_label
);
'''

client.query(agg_view).result() 
print("View created successfully!")

View created successfully!


In [21]:
run_query(f'''
select * from {DATASET}.state_agg
limit 10;
'''
)

dry-run: will scan 0.029 GB


,state_str,state,year,sector_code,sector_label,total_pay_st,total_emp_st,total_bs_st,avg_pay_st
0,Mississippi,28,2017,00,Total for all sectors,35366718.0,939485,59294,37.644793
1,Missouri,29,2017,00,Total for all sectors,116773811.0,2517204,150882,46.390285
2,Montana,30,2017,00,Total for all sectors,14773193.0,376565,38192,39.231455
3,Nebraska,31,2017,00,Total for all sectors,36415429.0,833472,54954,43.691245
4,Nevada,32,2017,00,Total for all sectors,50960493.0,1191625,66430,42.765545
5,New Hampshire,33,2017,00,Total for all sectors,30634092.0,603923,38371,50.725162
6,New Jersey,34,2017,00,Total for all sectors,220310440.0,3679443,233907,59.876030
7,New Mexico,35,2017,00,Total for all sectors,25723101.0,626466,44039,41.060650
8,Alabama,01,2017,00,Total for all sectors,71746399.0,1690061,100419,42.451958
9,Alaska,02,2017,00,Total for all sectors,15007844.0,262075,21279,57.265455


In [ ]:
create_ratio = f'''
create or replace table {DATASET}.ratio_df as (
  select
    c.*,
    s.total_pay_st,
    s.total_emp_st,
    s.total_bs_st,
    s.avg_pay_st,
    c.avg_pay_ct / nullif(s.avg_pay_st, 0) as pay_ratio
  from {DATASET}.county_agg c
  join {DATASET}.state_agg s
    on c.state   = s.state
    and c.year        = s.year
    and c.sector_code = s.sector_code
  where s.avg_pay_st is not null
    and c.avg_pay_ct is not null
);
'''

client.query(create_ratio).result()
print("Table ratio_df created successfully!")

Table ratio_df created successfully!


In [42]:
run_query(f'''
  select * from {DATASET}.ratio_df 
  order by pay_ratio desc 
  limit 10
''')

dry-run: will scan 0.055 GB


,county_str,county_code,state,sector_code,sector_label,year,num_employees,annual_payroll,num_businesses,avg_pay_ct,total_pay_st,total_emp_st,total_bs_st,avg_pay_st,pay_ratio
0,"Madison County, Montana",057,30,71,"Arts, entertainment, and recreation",2021,21,12253.0,19,583.476190,244380.0,9760,1313,25.038934,23.302756
1,"Montezuma County, Colorado",083,08,71,"Arts, entertainment, and recreation",2018,8,5514.0,4,689.250000,2038863.0,58569,3115,34.811299,19.799606
2,"Delaware County, Oklahoma",041,40,61,Educational services,2022,1,607.0,3,607.000000,834260.0,22281,864,37.442664,16.211453
3,"Dillingham Census Area, Alaska",070,02,31-33,Manufacturing,2019,11,9846.0,5,895.090909,749067.0,12334,555,60.731879,14.738403
4,"Grundy County, Iowa",075,19,53,Real estate and rental and leasing,2021,11,8038.0,6,730.727273,738748.0,14581,3518,50.665112,14.422691
5,"Grundy County, Iowa",075,19,53,Real estate and rental and leasing,2019,12,7104.0,5,592.000000,664043.0,15020,3297,44.210586,13.390458
6,"Dillingham Census Area, Alaska",070,02,31-33,Manufacturing,2018,16,10123.0,5,632.687500,581165.0,11692,537,49.706209,12.728541
7,"Montgomery County, Tennessee",125,47,11,"Agriculture, forestry, fishing and hunting",2023,1,625.0,3,625.000000,81169.0,1631,216,49.766401,12.558674
8,"Grundy County, Iowa",075,19,53,Real estate and rental and leasing,2020,16,8547.0,5,534.187500,682848.0,15139,3378,45.105225,11.843140
9,"Northumberland County, Virginia",133,51,11,"Agriculture, forestry, fishing and hunting",2019,18,11440.0,6,635.555556,227011.0,4226,638,53.717700,11.831399


In [20]:
import numpy as np
import plotly.express as px

### <a id="q1">Query 1</a>

Which counties significantly outperform their state sector pay?


We select the 100 most outperforming counties, filtering out the ones with $\geq100$ employees in order to exclude statistically unreliable small sample outliers (like a county with 5 employees in Arts with a 20x ratio because of one high-paid individual), and remove the sector `00`, which represent the aggregate value for all sectors.

In [19]:
q1 = run_query(f'''
select 
	county_str, 
	year, 
	sector_label, 
	num_employees,
	num_businesses,
    avg_pay_st,
    avg_pay_ct, 
	pay_ratio
from {DATASET}.ratio_df 
where num_employees >= 100 
and sector_code != '00'
order by pay_ratio desc 
limit 100;
'''
)

dry-run: will scan 0.040 GB


In [20]:
q1['county'] = q1.county_str.str.split(', ').str[0]
q1['state_str'] = q1.county_str.str.split(', ').str[1]

q1

,county_str,year,sector_label,num_employees,num_businesses,avg_pay_st,avg_pay_ct,pay_ratio,county,state_str
0,"Hall County, Georgia",2023,"Arts, entertainment, and recreation",873,82,44.010454,418.762887,9.515078,Hall County,Georgia
1,"Burke County, Georgia",2017,Construction,111,30,55.988215,483.045045,8.627620,Burke County,Georgia
2,"Hall County, Georgia",2022,"Arts, entertainment, and recreation",928,81,41.886246,318.766164,7.610282,Hall County,Georgia
3,"Hall County, Georgia",2017,"Arts, entertainment, and recreation",1100,77,36.910860,252.140000,6.831052,Hall County,Georgia
4,"Bristol Bay Borough, Alaska",2023,Manufacturing,101,9,72.690739,438.554455,6.033154,Bristol Bay Borough,Alaska
...,...,...,...,...,...,...,...,...,...,...
95,"Williams County, North Dakota",2017,"Arts, entertainment, and recreation",135,21,14.903953,46.681481,3.132154,Williams County,North Dakota
96,"Stevens County, Minnesota",2023,"Agriculture, forestry, fishing and hunting",145,11,60.721826,189.289655,3.117325,Stevens County,Minnesota
97,"Brown County, Wisconsin",2022,"Arts, entertainment, and recreation",3378,139,40.291052,125.287744,3.109567,Brown County,Wisconsin
98,"Nantucket County, Massachusetts",2018,Accommodation and food services,776,130,23.906547,74.228093,3.104927,Nantucket County,Massachusetts


In [30]:
fig = px.scatter(
    q1,
    x='avg_pay_st',
    y='avg_pay_ct',
    color='sector_label',
    size='pay_ratio',
    hover_data=['county', 'state_str', 'year'],
    title='County vs State Average Pay by Sector',
    labels={
        'avg_pay_st': 'State Avg Pay',
        'avg_pay_ct': 'County Avg Pay'
    }
)


fig.add_shape(
    type='line',
    x0=q1['avg_pay_st'].min(), y0=q1['avg_pay_st'].min(),
    x1=q1['avg_pay_st'].max(), y1=q1['avg_pay_st'].max(),
    line=dict(color='red', dash='dash')
)

fig.layout.update(showlegend=False)

fig.show()

- The red diagonal represents the situation where `pay_ratio == 1`. All our points are above it, they **systematically outperform** their state benchmark.
- The further above the line, the stronger the outliers.
- **Sector** dimension is encoded through colors.
- **Pay Ratio** is encoded with size.

In [ ]:
q1.sector_label.value_counts()
# More than half % of companies work in accomodation/food sector or Art

sector_label
Accommodation and food services                                             27
Arts, entertainment, and recreation                                         25
Manufacturing                                                                9
Transportation and warehousing                                               7
Administrative and support and waste management and remediation services     6
Professional, scientific, and technical services                             5
Other services (except public administration)                                4
Agriculture, forestry, fishing and hunting                                   4
Educational services                                                         4
Mining, quarrying, and oil and gas extraction                                3
Construction                                                                 2
Utilities                                                                    2
Retail trade                           

In [ ]:
q1.num_employees.quantile(np.arange(0.0, 1.1, 0.25))

# Half of our businesses have less than 260 people working there

0.00      100.0
0.25      145.0
0.50      258.5
0.75     800.25
1.00    11338.0
Name: num_employees, dtype: Float64

In [42]:
fig = px.scatter(
    q1,
    x='avg_pay_st',
    y='avg_pay_ct',
    color='sector_label',
    size='num_employees',
    hover_data=['county', 'state_str', 'year'],
    title='County vs State Average Pay by Sector - Size: Employees',
    labels={
        'avg_pay_st': 'State Avg Pay',
        'avg_pay_ct': 'County Avg Pay'
    }
)


fig.add_shape(
    type='line',
    x0=q1['avg_pay_st'].min(), y0=q1['avg_pay_st'].min(),
    x1=q1['avg_pay_st'].max(), y1=q1['avg_pay_st'].max(),
    line=dict(color='red', dash='dash')
)

fig.layout.update(showlegend=False)

fig.show()

In [58]:
q1.state_str.unique()

array(['Georgia', 'Alaska', 'Virginia', 'Michigan', 'Puerto Rico',
       'Louisiana', 'North Dakota', 'Mississippi', 'Missouri',
       'Massachusetts', 'Wisconsin', 'West Virginia', 'Pennsylvania',
       'Colorado', 'Alabama', 'Kansas', 'New Mexico', 'Oklahoma',
       'California', 'Minnesota'], dtype=object)

In [ ]:
q1.query("county == 'Hall County' and sector_label.str.contains('Art')").sort_values(by='year', ascending=False)

,county_str,year,sector_label,num_employees,num_businesses,avg_pay_st,avg_pay_ct,pay_ratio,county,state_str
0,"Hall County, Georgia",2023,"Arts, entertainment, and recreation",873,82,44.010454,418.762887,9.515078,Hall County,Georgia
2,"Hall County, Georgia",2022,"Arts, entertainment, and recreation",928,81,41.886246,318.766164,7.610282,Hall County,Georgia
41,"Hall County, Georgia",2021,"Arts, entertainment, and recreation",1995,81,43.758399,169.381955,3.870844,Hall County,Georgia
20,"Hall County, Georgia",2020,"Arts, entertainment, and recreation",2514,70,30.241321,131.285998,4.341279,Hall County,Georgia
47,"Hall County, Georgia",2019,"Arts, entertainment, and recreation",2563,74,34.883294,131.952009,3.782671,Hall County,Georgia
51,"Hall County, Georgia",2018,"Arts, entertainment, and recreation",2518,75,33.807771,123.809373,3.662157,Hall County,Georgia
3,"Hall County, Georgia",2017,"Arts, entertainment, and recreation",1100,77,36.910860,252.140000,6.831052,Hall County,Georgia


Hall County in Georgia, serves as a prominent filming destination for major tv and streaming series, largely driven by the region's diverse scenery and Georgia's robust production tax credits.

Example of productions made there:

1. [Tulsa King](https://en.wikipedia.org/wiki/Tulsa_King): 2022 $\leftrightarrow$  -
2. [Monarch](https://en.wikipedia.org/wiki/Monarch:_Legacy_of_Monsters): 2023 $\leftrightarrow$  -
3. [Ozark](https://en.wikipedia.org/wiki/Ozark_(TV_series)): 2017 $\leftrightarrow$ 2022 
4. [Murdaugh](https://en.wikipedia.org/wiki/Murdaugh:_Death_in_the_Family): 2025 

And much more.

For the complete list: https://www.exploregainesville.org/plan-your-visit/filmed-in-gainesville/

Specific findings:
- **Number of workers dropped over the years**: film productions use temporary crew, not permanent employees, when a show ends, those workers leave.

- **Average pay increased**: the permanent staff cover high-skilled jobs, like directors, producers, etc... Which have premium salaries.

- **Number of businesses stays more or less constant**: There are an handful of production companies and those numbers stay flat, there's no broad industry expansion.

In [ ]:
q1.query("sector_label.str.contains('food')").state_str.unique()

array(['Michigan', 'Alaska', 'North Dakota', 'Massachusetts'],
      dtype=object)

In [60]:
q1.query("state_str == 'Michigan' and sector_label.str.contains('food')")

,county_str,year,sector_label,num_employees,num_businesses,avg_pay_st,avg_pay_ct,pay_ratio,county,state_str
6,"Mackinac County, Michigan",2023,Accommodation and food services,538,98,23.538843,124.213755,5.276969,Mackinac County,Michigan
7,"Mackinac County, Michigan",2017,Accommodation and food services,575,101,17.018276,87.520000,5.142706,Mackinac County,Michigan
10,"Mackinac County, Michigan",2018,Accommodation and food services,586,100,17.935752,88.399317,4.928665,Mackinac County,Michigan
12,"Mackinac County, Michigan",2019,Accommodation and food services,584,99,18.598983,89.419521,4.807764,Mackinac County,Michigan
13,"Mackinac County, Michigan",2020,Accommodation and food services,596,99,15.238026,72.303691,4.744951,Mackinac County,Michigan
17,"Mackinac County, Michigan",2022,Accommodation and food services,624,101,22.447613,102.389423,4.561261,Mackinac County,Michigan
18,"Mackinac County, Michigan",2021,Accommodation and food services,553,105,21.954266,96.560579,4.398260,Mackinac County,Michigan


In [62]:
q1.query("state_str == 'Massachusetts' and sector_label.str.contains('food')")

,county_str,year,sector_label,num_employees,num_businesses,avg_pay_st,avg_pay_ct,pay_ratio,county,state_str
29,"Nantucket County, Massachusetts",2020,Accommodation and food services,553,120,19.820959,80.041591,4.038230,Nantucket County,Massachusetts
30,"Nantucket County, Massachusetts",2023,Accommodation and food services,624,116,32.905355,131.616987,3.999865,Nantucket County,Massachusetts
35,"Nantucket County, Massachusetts",2022,Accommodation and food services,574,116,31.078606,122.831010,3.952269,Nantucket County,Massachusetts
37,"Nantucket County, Massachusetts",2021,Accommodation and food services,522,121,31.753352,124.197318,3.911314,Nantucket County,Massachusetts
57,"Nantucket County, Massachusetts",2019,Accommodation and food services,664,132,25.693039,88.861446,3.458581,Nantucket County,Massachusetts
64,"Dukes County, Massachusetts",2022,Accommodation and food services,566,154,31.078606,105.042403,3.379894,Dukes County,Massachusetts
81,"Dukes County, Massachusetts",2021,Accommodation and food services,549,152,31.753352,103.112933,3.247309,Dukes County,Massachusetts
84,"Nantucket County, Massachusetts",2017,Accommodation and food services,742,125,23.050257,73.994609,3.210142,Nantucket County,Massachusetts
85,"Dukes County, Massachusetts",2020,Accommodation and food services,537,135,19.820959,63.607076,3.209082,Dukes County,Massachusetts
98,"Nantucket County, Massachusetts",2018,Accommodation and food services,776,130,23.906547,74.228093,3.104927,Nantucket County,Massachusetts


All these are high end luxury/touristic places, with [several bilionaires there](https://www.realtor.com/news/trends/nantucket-summer-millionaires-taking-over/) that rump up prices up, and conversely pays. 

These places can offer very high [pays](https://www.bostonmagazine.com/news/2025/08/19/nantucket-food-insecurity/), especially comparared to other, non luxury places in other part of the Country.


In [70]:
q1.query("state_str == 'Alaska' and sector_label.str.contains('food')")

,county_str,year,sector_label,num_employees,num_businesses,avg_pay_st,avg_pay_ct,pay_ratio,county,state_str
9,"Denali Borough, Alaska",2022,Accommodation and food services,154,38,34.694692,171.902597,4.954723,Denali Borough,Alaska
11,"Denali Borough, Alaska",2017,Accommodation and food services,174,38,27.512868,135.362069,4.919955,Denali Borough,Alaska
16,"Denali Borough, Alaska",2019,Accommodation and food services,180,38,29.073965,134.344444,4.620782,Denali Borough,Alaska
21,"Denali Borough, Alaska",2023,Accommodation and food services,183,35,36.338560,156.934426,4.318675,Denali Borough,Alaska
31,"Denali Borough, Alaska",2018,Accommodation and food services,172,37,28.693444,114.197674,3.979922,Denali Borough,Alaska


In the specific case of Alaska, the effect is brought by Denali National Park and Preserve, and seasonal works, with 60+ hour work weeks during the mid-May to mid-September time of the year.

In [4]:
run_query(f'''
select 
    sector_code, 
    sector_label, 
    avg(avg_pay_st) as avg_state_sector 
from {DATASET}.state_agg
where year = 2023
and sector_code != '00'
group by sector_code, sector_label 
order by avg_state_sector;
'''
)

dry-run: will scan 0.027 GB


,sector_code,sector_label,avg_state_sector
0,72,Accommodation and food services,25.910427
1,44-45,Retail trade,35.803216
2,71,"Arts, entertainment, and recreation",38.009862
3,81,Other services (except public administration),40.844677
4,61,Educational services,41.818964
5,56,Administrative and support and waste managemen...,48.399369
6,11,"Agriculture, forestry, fishing and hunting",55.546530
7,48-49,Transportation and warehousing,57.556139
8,62,Health care and social assistance,60.677441
9,53,Real estate and rental and leasing,60.909456


As we can see both Art and Accomodation sectors are some of the least paying sectors in 2023. 

**Conclusion:** the large pay-ratio that our data exibiths (over 50% of the top 100 counties), are given by enourmous inequalities in those sectors (a handful of people earning 10x more than the average worker), concentrated in very specific geographical places, such as high end tourist resorts. 

High paying sectors like finance or utilities don't appear (or are a small minority) in `q1` results. This happens because there's less inequality: people working there earn a higher baseline, bringing the pay_ratio value down.

### <a id="q2">Query 2</a>

Which sectors saw the biggest drop in average county pay because of COVID?

We compute the year-over-year percentage change in average `avg_pay_ct` per sector, using the `LAG` to compare each year to the previous one. Also here, we filter out entries with less than 100 employees and ignore sector_code `00`

In [13]:
create_q2base = f'''
create or replace view {DATASET}.q2_base as (
  select
    sector_code,
    sector_label,
    year,
    avg(avg_pay_ct) as avg_pay
  from {DATASET}.ratio_df
  where sector_code != '00'
    and num_employees >= 100
  group by sector_code, sector_label, year
);
'''

client.query(create_q2base).result()

create_q2 = f'''
create or replace view {DATASET}.q2 as (
  select
    sector_code,
    sector_label,
    year,
    avg_pay,
    lag(avg_pay, 1) over (partition by sector_code order by year) as prev_avg_pay,
    ((avg_pay - lag(avg_pay, 1) over (partition by sector_code order by year)) 
      / lag(avg_pay, 1) over (partition by sector_code order by year)) * 100 as p_change
  from {DATASET}.q2_base
);
'''

client.query(create_q2).result()
print("View computed")

View computed


In [17]:
q2 = run_query(f'''select sector_label, p_change from {DATASET}.q2
where prev_avg_pay is not null  
  and year = 2020
  and p_change < 0
order by p_change, year 
;
'''
)
q2

dry-run: will scan 0.022 GB


,sector_label,p_change
0,"Arts, entertainment, and recreation",-12.942948
1,"Mining, quarrying, and oil and gas extraction",-9.715025
2,Accommodation and food services,-8.710170
3,Educational services,-0.296115
4,Construction,-0.250702
5,Other services (except public administration),-0.221278
6,Real estate and rental and leasing,-0.030529


In [21]:
fig = px.bar(
    q2,
    x='sector_label',
    y='p_change',
    title='Sectors with Declining Pay Ratio in 2021 (Post-COVID Effect)',
    labels={
        'sector_label': 'Sector',
        'p_change': 'Pay Ratio Change (%)'
    },
    color='p_change',
    color_continuous_scale='Reds_r', 
    text='p_change'
)

fig.update_traces(
    texttemplate='%{text:.2f}%',
    textposition='outside'
)

fig.update_layout(
    xaxis_title='',
    yaxis_title='Pay Ratio Change (%)',
    coloraxis_showscale=False,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
    showlegend=False,
    shapes=[dict(
        type='line',
        x0=-0.5, x1=len(q2)-0.5,
        y0=0, y1=0,
        line=dict(color='black', width=1)
    )]
)

fig.update_layout(
    autosize=False,
    width=1800,
    height=500,
)
fig.update_xaxes(showticklabels=False)



fig.show()

The top three hardest-hit sectors in 2020 tell a coherent story with research and data gathered from other sources:

- **Arts & Entertainment**: performing arts, festivals, and live events faced near-total shutdown. According to [public reports](https://www.arts.gov/news/press-releases/2022/new-data-show-economic-impact-covid-19-arts-culture-sector), the value added by performing arts presenters fell by nearly 73% between 2019 and 2020, with artist unemployment reaching 10.3%.

- **Mining & Oil**: heavily dependent on global industrial demand, which collapsed as factories shut down worldwide. As noted in [The economic impact of Covid-19 on the mining industry](https://www.sciencedirect.com/science/article/pii/S2214790X2030126X):
  > Dramatic contraction in demand as industrial production and construction declined [...] 
  > causing dramatic falls in the prices of a range of metals and minerals.

- **Accommodation & Food Services**: direct consequence of travel restrictions and lockdowns, consistent with findings in [Tourism: The Great Patient of COVID-2019](https://www.researchgate.net/publication/340771406_Tourism_The_Great_Patient_of_Coronavirus_COVID-2019):
  > In cases of pandemics, tourists cancel their travels [...] 
  > Tourism is currently one of the most affected sectors.

- Educational services, Construction and Other services show minimal declines (-0.03% to -0.30%), they either shifted to remote delivery or maintained essential operations throughout 2020.

### <a id="q3">Query 3</a>

How salaries have changed across different business size between 2017-2023

We create a wage growth index anchored at 2017 = 100, separately for each business size bucket. This lets us compare *growth trajectories* across sizes on the same scale, regardless of absolute pay differences (large businesses always pay more in absolute terms the index shows who grew *fastest*).

We use state-level data, filtering to:
- `lfo == '001'` (all legal forms aggregate)
- `bsize_code != '001'` (exclude the "All establishments" total)
- `sector_code == '00'` (all sectors aggregate)

In [129]:
create_q3base = f'''
create or replace view {DATASET}.q3_base as (
  select
    year,
    bsize,
    bsize_code,
    avg(annual_payroll / nullif(num_employees, 0)) as avg_salary
  from {DATASET}.cbp_state
  where lfo = '001'
    and bsize_code != '001'
    and sector_code = '00'
    and annual_payroll > 0
    and num_employees > 0
  group by year, bsize, bsize_code
);
'''

client.query(create_q3base).result()

create_q3 = f'''
create or replace view {DATASET}.q3 as (
  select
    year,
    bsize,
    bsize_code,
    avg_salary,
    first_value(avg_salary) over (
      partition by bsize_code 
      order by year 
      rows between unbounded preceding and unbounded following
    ) as base_2017
  from {DATASET}.q3_base
);
'''

client.query(create_q3).result()

print("View computed")

View computed


In [106]:
q3 = run_query(f'''
select
  year,
  bsize,
  bsize_code,
  avg_salary,
  round((avg_salary / base_2017) * 100, 2) as idx
from {DATASET}.q3
order by bsize_code, year;
'''
)

q3

dry-run: will scan 0.024 GB


,year,bsize,bsize_code,avg_salary,idx
0,2017,Establishments with less than 5 employees,210,47.304655,100.00
1,2018,Establishments with less than 5 employees,210,48.778122,103.11
2,2019,Establishments with less than 5 employees,210,49.648767,104.96
3,2020,Establishments with less than 5 employees,210,49.644976,104.95
4,2021,Establishments with less than 5 employees,210,56.364814,119.15
...,...,...,...,...,...
58,2019,"Establishments with 1,000 employees or more",260,68.217549,106.86
59,2020,"Establishments with 1,000 employees or more",260,68.408041,107.15
60,2021,"Establishments with 1,000 employees or more",260,75.593298,118.41
61,2022,"Establishments with 1,000 employees or more",260,78.319751,122.68


In [107]:
bsize_short = {
    'Establishments with less than 5 employees': '<5 emp',
    'Establishments with 5 to 9 employees': '5-9 emp',
    'Establishments with 10 to 19 employees': '10-19 emp',
    'Establishments with 20 to 49 employees': '20-49 emp',
    'Establishments with 50 to 99 employees': '50-99 emp',
    'Establishments with 100 to 249 employees': '100-249 emp',
    'Establishments with 250 to 499 employees': '250-499 emp',
    'Establishments with 500 to 999 employees': '500-999 emp',
    'Establishments with 1,000 employees or more': '1000+ emp'
}

q3['bsize_short'] = q3['bsize'].map(bsize_short)

fig = px.line(
    q3,
    x='year',
    y='idx',
    color='bsize_short',
    title='Pay Trend Index by Business Size (2017 = 100)',
    labels={
        'index': 'Pay Index (2017 = 100)',
        'year': 'Year',
        'bsize_short': 'Business Size'
    },
    category_orders={'bsize_short': list(bsize_short.values())} 
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
    xaxis=dict(gridcolor='lightgrey'),
    legend=dict(
        title='Business Size',
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01
    )
)

fig.add_vline(
    x=2020,
    line_dash='dash',
    line_color='red',
    annotation_text='COVID-19',
    annotation_position='top right'
)

fig.show()

In [122]:
q3.query("year == 2023")

,year,bsize,bsize_code,avg_salary,idx,bsize_short
6,2023,Establishments with less than 5 employees,210,58.538748,123.75,<5 emp
13,2023,Establishments with 5 to 9 employees,220,48.722436,127.04,5-9 emp
20,2023,Establishments with 10 to 19 employees,230,49.317292,126.65,10-19 emp
27,2023,Establishments with 20 to 49 employees,241,50.970594,128.30,20-49 emp
34,2023,Establishments with 50 to 99 employees,242,55.970217,130.80,50-99 emp
41,2023,Establishments with 100 to 249 employees,251,62.105250,132.79,100-249 emp
48,2023,Establishments with 250 to 499 employees,252,70.174780,131.61,250-499 emp
55,2023,Establishments with 500 to 999 employees,254,78.076018,129.99,500-999 emp
62,2023,"Establishments with 1,000 employees or more",260,81.082509,127.01,1000+ emp


In [121]:
iqr = q3.query("year == 2023").idx.quantile(0.75) - q3.query("year == 2023").idx.quantile(0.25)

print(f'{np.round(iqr, 2)}%')

3.79%


The wage growth trajectory is remarkably uniform across all business sizes: all lines track closely together from 2017 through 2023, with an IQR of just  **3.79%** across size categories in 2023. This means the pandemic did not structurally widen or narrow the wage gap between small and large businesses.

Two things stand out:

1. **The COVID dip is visible but not sharp**
    All size categories show a flattening trend in 2020, but no collapse is present (Probably effects of [PPP Loan Programs](https://www.sba.gov/funding-programs/loans/covid-19-relief-options/paycheck-protection-program) reducing shock across businesses of all sizes). The recovery from 2021 onward is steep and sustained, surpassing pre-COVID trajectories for all categories.

2. **A U-shaped index distribution in 2023**
    Looking at the final year, mid-sized businesses (from 50 to 500 employees) reach the highest indexes, while on the tails (both micro and large businesses show more modest growth).

We might expect large corporations to dominate wage growth due to economies of scale, or small businesses to be most flexible to shocks and recoveries. Instead, it apears that the mid-business segment had competitive pressure to raise wages which consistent with research on the matter, concering the so called ['Great Resignation'](https://sloanreview.mit.edu/article/toxic-culture-is-driving-the-great-resignation/) started in 2021. 

### <a id="q4">Query 4</a>

Which sectors have the highest concentration of corporate, sole-proprietorship/partnership, government/no-profit jobs/businesses?

We classify establishments by legal form of organization (LFO) into three groups:
- **Corporate** (`9101` C-corps, `9111` S-corps)
- **Sole/Partnership** (`920` proprietorships, `930` partnerships)
- **Gov/Nonprofit** (`932` nonprofits, `933` government)
- **Other** (remainder)

We run this analysis twice:
1. **By number of businesses** — who *owns* most entities?
2. **By number of employees** — who *employs* most workers?

In [128]:
create_q4 = f'''
create or replace view {DATASET}.q4 as (
  select
    sector_code,
    sector_label,
    year,
    sum(case when lfo in ('9101', '9111') then num_businesses else 0 end) as corporate,
    sum(case when lfo in ('920', '930')   then num_businesses else 0 end) as sole,
    sum(case when lfo in ('932', '933')   then num_businesses else 0 end) as gov_np,
    sum(num_businesses)                                                    as total_b
  from {DATASET}.cbp_state
  where bsize_code = '001'
    and sector_code != '00'
  group by sector_code, sector_label, year
);
'''

client.query(create_q4).result()

create_q4_2 = f'''
create or replace view {DATASET}.q4_2 as (
  select
    sector_code,
    sector_label,
    year,
    sum(case when lfo in ('9101', '9111') then num_employees else 0 end) as corporate,
    sum(case when lfo in ('920', '930')   then num_employees else 0 end) as sole,
    sum(case when lfo in ('932', '933')   then num_employees else 0 end) as gov_np,
    sum(num_employees) as total_emp
  from {DATASET}.cbp_state
  where bsize_code = '001'
    and sector_code != '00'
  group by sector_code, sector_label, year
);
'''

client.query(create_q4_2).result()

print("View computed")

View computed


In [130]:
q4 = run_query(f'''
select
  sector_code,
  sector_label,
  year,
  corporate,
  sole,
  gov_np,
  total_b,
  round(corporate / nullif(total_b, 0) * 100, 2) as corp_share,
  round(sole     / nullif(total_b, 0) * 100, 2) as sole_share,
  round(gov_np   / nullif(total_b, 0) * 100, 2) as gov_np_share
from {DATASET}.q4
where year = 2023
order by sole_share desc;
'''
)


q4_2 = run_query(f'''
select
  sector_code,
  sector_label,
  year,
  corporate,
  sole,
  gov_np,
  total_emp,
  round(corporate / nullif(total_emp, 0) * 100, 2) as corp_share,
  round(sole     / nullif(total_emp, 0) * 100, 2) as sole_share,
  round(gov_np   / nullif(total_emp, 0) * 100, 2) as gov_np_share
from {DATASET}.q4_2
where year = 2023
order by sole_share desc;
'''
)


dry-run: will scan 0.019 GB
dry-run: will scan 0.019 GB


In [131]:
q4

,sector_code,sector_label,year,corporate,sole,gov_np,total_b,corp_share,sole_share,gov_np_share
0,72,Accommodation and food services,2023,503264,283187,4301,1583823,31.78,17.88,0.27
1,11,"Agriculture, forestry, fishing and hunting",2023,15072,7229,300,45493,33.13,15.89,0.66
2,53,Real estate and rental and leasing,2023,336539,127239,5392,944042,35.65,13.48,0.57
3,21,"Mining, quarrying, and oil and gas extraction",2023,17505,5760,3,46757,37.44,12.32,0.01
4,56,Administrative and support and waste managemen...,2023,336499,109426,7203,907253,37.09,12.06,0.79
5,71,"Arts, entertainment, and recreation",2023,98086,39943,28623,334063,29.36,11.96,8.57
6,31-33,Manufacturing,2023,226860,58657,309,573244,39.57,10.23,0.05
7,23,Construction,2023,651500,165584,832,1636063,39.82,10.12,0.05
8,44-45,Retail trade,2023,840250,200014,10901,2109872,39.82,9.48,0.52
9,52,Finance and insurance,2023,359883,89100,21164,942919,38.17,9.45,2.24


In [ ]:
q4['other_share'] = 100 - q4['corp_share'] - q4['sole_share'] - q4['gov_np_share']

q4['sector_short'] = q4['sector_label'].str[:25]

from plotly.subplots import make_subplots
import plotly.graph_objects as go

n = len(q4)
cols = 4
rows = (n + cols - 1) // cols

specs = [[{'type': 'pie'} for _ in range(cols)] for _ in range(rows)]
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=q4['sector_short'].tolist(),
    specs=specs
)

colors = ['#636EFA', '#EF553B', '#00CC96', '#AAAAAA'] 
labels = ['Corporate', 'Sole/Partnership', 'Gov/Nonprofit', 'Other']

for i, row_data in q4.iterrows():
    r = i // cols + 1
    c = i % cols + 1
    values = [
        row_data['corp_share'],
        row_data['sole_share'],
        row_data['gov_np_share'],
        row_data['other_share']
    ]
    fig.add_trace(
        go.Pie(
            labels=labels,
            values=values,
            name=row_data['sector_short'],
            showlegend=(i == 0),
            marker=dict(colors=colors),
            textinfo='percent',
            hole=0.3 
        ),
        row=r, col=c
    )

fig.update_layout(
    title='Business Ownership Structure by Sector (2023)',
    height=300 * rows,
    plot_bgcolor='white'
)

fig.show()

In [ ]:
q4_2['other_share'] = 100 - q4_2['corp_share'] - q4_2['sole_share'] - q4_2['gov_np_share']

q4_2['sector_short'] = q4_2['sector_label'].str[:25]

from plotly.subplots import make_subplots
import plotly.graph_objects as go

n = len(q4_2)
cols = 4
rows = (n + cols - 1) // cols

specs = [[{'type': 'pie'} for _ in range(cols)] for _ in range(rows)]
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=q4_2['sector_short'].tolist(),
    specs=specs
)

colors = ['#636EFA', '#EF553B', '#00CC96', '#AAAAAA'] 
labels = ['Corporate', 'Sole/Partnership', 'Gov/Nonprofit', 'Other']

for i, row_data in q4_2.iterrows():
    r = i // cols + 1
    c = i % cols + 1
    values = [
        row_data['corp_share'],
        row_data['sole_share'],
        row_data['gov_np_share'],
        row_data['other_share']
    ]
    fig.add_trace(
        go.Pie(
            labels=labels,
            values=values,
            name=row_data['sector_short'],
            showlegend=(i == 0),
            marker=dict(colors=colors),
            textinfo='percent',
            hole=0.3 
        ),
        row=r, col=c
    )

fig.update_layout(
    title='Employment Share by Ownership Type and Sector (2023)',
    height=300 * rows,
    plot_bgcolor='white'
)

fig.show()

1. Corporate entities dominate across all sectors, while sole proprietorships remain limited in most industries. Gov/Nonprofit organizations have meaningful presence only in a few sectors: **Education (15.2%)** **Health Care (9.1%)**, **Utilities (3.8%)** and **Other Services (11.7%)**.

2. The picture shifts in favour of Gov/Nonprofit in those same sectors:
    - **Education**: 15.2% of businesses $\rightarrow$ **38.6% of employment**
    - **Health Care**: 9.1% of businesses $\rightarrow$ **19.5% of employment**  
    - **Other Services**: 11.7% of businesses $\rightarrow$ **22.4% of employment**

This divergence shows structural differences in organization size: corporate entities are more *numerous*, Gov/Nonprofit organizations are significantly *larger* on average (a single public university or hospital  employs far more people than a typical private tutoring center or clinic).

On the other hand, sectors like **Mining**, **Construction** and **Wholesale Trade** show almost no Gov/Nonprofit presence in either chart. They are entirely private-sector dominated with negligible-to-none public or nonprofit participation.

### <a id="q5">Query 5</a>

Which counties saw a net growth in business establishments despite shrinking payrolls?

We identify counties where the number of establishments *grew* year-over-year while average pay *fell*, which is a pattern consistent with:

- **Business fragmentation**: large employers replaced by many smaller ones

- **Sector shift**: low-wage industries expanding while high-wage ones contract

- **Gig economy effects**: more independent contractors registered as businesses

We filter `tot_businesses >= 500` to focus on counties large enough for the trend to be statistically significant (smaller counties can show 20%+ changes from just a handful of businesses opening or closing).

In [140]:
create_q5_base = f'''
create or replace view {DATASET}.q5_base as (
  select
    state,
    county_code,
    county_str,
    year,
    avg(annual_payroll / nullif(num_employees, 0)) as avg_pay,
    sum(num_businesses) as tot_businesses
  from {DATASET}.cbp_county
  where num_employees > 0
  group by state, county_code, county_str, year
);
'''

client.query(create_q5_base).result()

create_q5 = f'''
create or replace view {DATASET}.q5 as (
  select
    state,
    county_code,
    county_str,
    year,
    avg_pay,
    tot_businesses,
    lag(avg_pay, 1)        over (partition by county_code, state order by year) as prev_avg_pay,
    lag(tot_businesses, 1) over (partition by county_code, state order by year) as prev_tot_businesses
  from {DATASET}.q5_base
);
'''

client.query(create_q5).result()

print("View computed")

View computed


In [22]:
q5 = run_query(f'''
select
  county_str,
  year,
  tot_businesses,
  round(((avg_pay - prev_avg_pay) / nullif(prev_avg_pay, 0)) * 100, 2)                     as salary_var,
  round(((tot_businesses - prev_tot_businesses) / nullif(prev_tot_businesses, 0)) * 100, 2) as b_var
from {DATASET}.q5
where prev_avg_pay is not null
  and ((avg_pay - prev_avg_pay) / nullif(prev_avg_pay, 0)) * 100 < 0
  and ((tot_businesses - prev_tot_businesses) / nullif(prev_tot_businesses, 0)) * 100 > 0
  and tot_businesses >= 500
order by b_var desc;
'''
)
q5

dry-run: will scan 0.025 GB


,county_str,year,tot_businesses,salary_var,b_var
0,"Sheridan County, Wyoming",2022,4537,-8.11,17.63
1,"Barranquitas Municipio, Puerto Rico",2022,629,-0.20,13.54
2,"Mariposa County, California",2022,713,-0.43,12.46
3,"Fayette County, Tennessee",2021,1386,-1.35,10.79
4,"Warren County, Missouri",2018,1254,-5.48,10.68
...,...,...,...,...,...
1277,"Westchester County, New York",2018,63490,-2.32,0.03
1278,"Montrose County, Colorado",2023,2996,-1.09,0.03
1279,"Lapeer County, Michigan",2023,3535,-0.69,0.03
1280,"Middlesex County, New Jersey",2019,43654,-5.56,0.03


In [146]:
fig = px.scatter(
    q5,
    x='salary_var',
    y='b_var',
    size='tot_businesses',
    color='year',
    hover_data=['county_str', 'year', 'tot_businesses'],
    title='Counties with Business Growth Despite Wage Decline',
    labels={
        'salary_var': 'Wage Change (%)',
        'b_var':      'Business Growth (%)',
        'tot_businesses': 'Total Businesses',
        'year': 'Year'
    },
    color_continuous_scale='Viridis',
    size_max=40
)

fig.update_traces(textposition='top center', textfont_size=8)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey', title='Business Growth (%)'),
    xaxis=dict(gridcolor='lightgrey', title='Wage Change (%)'),
    shapes=[
        dict(type='line', x0=q5['salary_var'].min(), x1=0,
             y0=0, y1=0, line=dict(color='red', dash='dash')),
        dict(type='line', x0=0, x1=0,
             y0=0, y1=q5['b_var'].max(), line=dict(color='red', dash='dash'))
    ]
)

fig.show()

In [24]:
q5_filtered = q5[q5['b_var'] >= 2].copy() 

top_n = 10
top_idx = q5_filtered.nlargest(top_n, 'b_var').index
q5_filtered['label'] = ''
q5_filtered.loc[top_idx, 'label'] = q5_filtered.loc[top_idx, 'county_str']

fig = px.scatter(
    q5_filtered,
    x='salary_var',
    y='b_var',
    size='tot_businesses',
    color='year',
    hover_data=['county_str', 'year', 'tot_businesses', 'salary_var', 'b_var'],
    text='label',
    title='Counties with Business Growth Despite Wage Decline',
    labels={
        'salary_var': 'Wage Change (%)',
        'b_var': 'Business Growth (%)',
        'tot_businesses': 'Total Businesses',
        'year': 'Year'
    },
    color_continuous_scale='Viridis',
    size_max=40
)

fig.update_traces(
    textposition='top center',
    textfont_size=9,
    marker=dict(opacity=0.7) 
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
    xaxis=dict(gridcolor='lightgrey', range=[-25, 2]),
    shapes=[
        dict(type='line', x0=-25, x1=0, y0=0, y1=0,
             line=dict(color='red', dash='dash')),
        dict(type='line', x0=0, x1=0, y0=0, y1=q5_filtered['b_var'].max(),
             line=dict(color='red', dash='dash'))
    ]
)

fig.show()

In [55]:
q5_filtered['state'] = q5_filtered.label.str.split(', ').str[1]
q5_filtered.value_counts('state')

state
Puerto Rico    4
Georgia        2
Wyoming        1
California     1
Tennessee      1
Missouri       1
Name: count, dtype: int64

In [51]:
q5_filtered['tot_state'] = q5_filtered.county_str.str.split(', ').str[1]
(q5_filtered.value_counts('tot_state') / len(q5_filtered)) * 100


tot_state
Texas                                           10.318949
Puerto Rico                                      8.630394
Georgia                                          8.630394
Tennessee                                        4.878049
Virginia                                         4.127580
North Carolina                                   3.939962
Florida                                          3.752345
Michigan                                         3.752345
Iowa                                             3.189493
Utah                                             3.001876
Colorado                                         3.001876
Missouri                                         2.814259
Indiana                                          2.814259
California                                       2.626642
Idaho                                            2.439024
South Carolina                                   2.251407
Arkansas                                         2.251407
Kent

While Texas appears most frequently, because of its number of counties, most of the most extreme values are located in Puerto Rico, while the most frequent state is Texas. This is due to the effect of [Huricane Maria](https://edition.cnn.com/specials/weather/hurricane-maria), happened in 2017 (at the start of our dataset)

As stated in the [Entrepreneurial dynamics in Puerto Rico before and after Hurricane Maria](https://www.researchgate.net/publication/348575665_Entrepreneurial_dynamics_in_Puerto_Rico_before_and_after_Hurricane_Maria) paper, the immediate aftermath of the disaster caused widespread establishment closures and severe employment contraction. This was followed by a resilient rebound characterized by a surge of new micro-enterprise.

This reflects clear pattern of business fragmentation: we have large company closing or downsizing, dragging down the average payroll, while a proliferation of smaller, lower-paying businesses took their place and drove up the total business count up.

## <a id="MLintro">Machine Learning Pipeline</a>

In [13]:
mod_a = f'''
create or replace model {DATASET}.model_a
options(
  model_type = 'linear_reg',
  input_label_cols = ['pay_ratio'],
  data_split_method = 'no_split'
) as
select
  sector_code,
  year,
  num_businesses,
  num_employees,
  state,
  county_code,
  pay_ratio
from {DATASET}.ratio_df
where num_employees >= 100
  and num_businesses >= 100
  and year <= 2021;
'''

client.query(mod_a).result()
print('Train Complete')

Train Complete


In [16]:
eval_mod_a = f'''
select *
from ml.evaluate(
  model {DATASET}.model_a,
  (
    select
      sector_code, year, num_businesses,
      num_employees, state, county_code, pay_ratio
    from {DATASET}.ratio_df
    where num_employees >= 100
      and num_businesses >= 100
      and year > 2021
  )
);
'''
eval_result = client.query(eval_mod_a).result().to_dataframe()
display(eval_result)


,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,0.137994,0.06488,0.008719,0.095845,-0.448229,-0.445826


It wasn't possible to use BigQueryML Regression Forest due to the following error:

> An internal error occurred and the request could not be completed. This is usually caused by a transient issue. 
> Retrying the job with back-off as described in the BigQuery SLA should solve the problem: https://cloud.google.com/bigquery/sla. 
> If the error continues to occur please contact support at https://cloud.google.com/support. Error: 80038528

Even after trying several times, in the span of hours.

As a fallback, a **Linear Regression** model was trained, which produced significantly worse results than spark.

The fact that **$R^2$** is negative means our linear model is worse at predicting than a simple horizontal line at `mean(pay_ratio)`. This is something expected, since we have seen that `pay_ratio` has a **non-linear, right-skewed distribution**, and linear regression models are very sensitive to outliers.

## <a id="#conc">Conclusion</a>

The queries give us a complete picture about **geographic and structural wage inequality** across US counties:

- **Sector and geography are the primary drivers of wage outliers**.
  Counties that significantly outperform their state benchmark are not random but cluster around specific economic drivers: from film production tax incentives to luxury tourism economies. These patterns repeat consistently across multiple years, confirming structural rather than accidental causes.

- **COVID's wage impact was not as sharp as expected in the long run** 
  While the sharpest absolute wage drops hit Arts & Entertainment (-12.9%) and Mining (-9.7%) in 2020, the long term trend observed by Q3 showed us that it did not widen wage inequality between large and small companies, and was recovered immediately

- **Corporate entities dominate by count, but Gov/Nonprofit dominates by size** 
  In Education, Health Care and Other Services, nonprofit and government organizations represent a minority of business entities but employ a disproportionate share of the workforce, revealing structural size asymmetries between ownership types.

- **Business fragmentation and wage decline co-occur in specific counties** 
  Counties showing simultaneous business growth and wage decline are concentrated in rural areas or Puerto Rico consistent with the events and research on the matter.

- **Prediction (Colab)** 
  Sector (37%) and geography (27%) are the strongest predictors of pay ratio, while year carries virtually no signal, confirming that wage outliers are structurally determined by geographical or sectorial inequality (Q1) rather than temporal. Wage growth direction is primarily predicted by year (86%), reflecting macro-economic cycles that affect all geographies simultaneously.